In [1]:

# Practical 4: Molecular Docking and Virtual Screening

# Suppress warning messages
import warnings
warnings.filterwarnings("ignore")

import os
import shutil
import subprocess
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski


# PART 1: VIRTUAL SCREENING

# Step 1: Create a list of molecules
ligands = {
    "Molecule": [
        "Ligand_01",
        "Ligand_02",
        "Ligand_03",
        "Ligand_04",
        "Ligand_05",
        "Ligand_06"
    ],

    "SMILES": [
        "CCO",
        "CC(=O)Oc1ccccc1C(=O)O",
        "CCN(CC)CC",
        "c1ccccc1",
        "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
        "CCOC(=O)c1ccccc1"
    ]
}

ligands_df = pd.DataFrame(ligands)

print("Molecule List:")
print(ligands_df)


# Step 2: Calculate molecular properties
results = []

for index, row in ligands_df.iterrows():

    molecule = Chem.MolFromSmiles(row["SMILES"])

    if molecule is not None:

        results.append({
            "Molecule": row["Molecule"],
            "Molecular_Weight": round(Descriptors.MolWt(molecule), 2),
            "LogP": round(Descriptors.MolLogP(molecule), 2),
            "HBD": Lipinski.NumHDonors(molecule),
            "HBA": Lipinski.NumHAcceptors(molecule)
        })


# Step 3: Create a DataFrame
desc = pd.DataFrame(results)

print("\nMolecular Properties:")
print(desc)


# Step 4: Apply Lipinski's Rule of Five
filtered = desc[
    (desc["Molecular_Weight"] <= 500) &
    (desc["LogP"] <= 5) &
    (desc["HBD"] <= 5) &
    (desc["HBA"] <= 10)
]

print("\nVirtual Screening Results:")
print(filtered)


# PART 2: MOLECULAR DOCKING

# Step 5: Check AutoDock Vina
vina = shutil.which("vina")

# Protein and ligand files
receptor = "receptor.pdbqt"
ligand = "ligand_01.pdbqt"

# Docking box centre
center_x = 0
center_y = 0
center_z = 0

# Docking box size
size_x = 20
size_y = 20
size_z = 20


# Step 6: Check required files
if vina and os.path.exists(receptor) and os.path.exists(ligand):

    print("\nStarting Molecular Docking...")

    command = [
        vina,
        "--receptor", receptor,
        "--ligand", ligand,
        "--center_x", str(center_x),
        "--center_y", str(center_y),
        "--center_z", str(center_z),
        "--size_x", str(size_x),
        "--size_y", str(size_y),
        "--size_z", str(size_z),
        "--out", "docked_ligand.pdbqt"
    ]

    # Step 7: Execute docking
    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("Docking completed successfully.")
        print(result.stdout)
        print("Docked structure saved as docked_ligand.pdbqt")

    else:
        print("Docking failed.")
        print(result.stderr)

else:
    print("\nDocking cannot be performed.")
    print("Check AutoDock Vina and the prepared PDBQT files.")


Molecule List:
    Molecule                      SMILES
0  Ligand_01                         CCO
1  Ligand_02       CC(=O)Oc1ccccc1C(=O)O
2  Ligand_03                   CCN(CC)CC
3  Ligand_04                    c1ccccc1
4  Ligand_05  CC(C)Cc1ccc(cc1)C(C)C(=O)O
5  Ligand_06            CCOC(=O)c1ccccc1

Molecular Properties:
    Molecule  Molecular_Weight  LogP  HBD  HBA
0  Ligand_01             46.07 -0.00    1    1
1  Ligand_02            180.16  1.31    1    3
2  Ligand_03            101.19  1.35    0    1
3  Ligand_04             78.11  1.69    0    0
4  Ligand_05            206.28  3.07    1    1
5  Ligand_06            150.18  1.86    0    2

Virtual Screening Results:
    Molecule  Molecular_Weight  LogP  HBD  HBA
0  Ligand_01             46.07 -0.00    1    1
1  Ligand_02            180.16  1.31    1    3
2  Ligand_03            101.19  1.35    0    1
3  Ligand_04             78.11  1.69    0    0
4  Ligand_05            206.28  3.07    1    1
5  Ligand_06            150.18  1.86